# 02 Census hierarchy

The UBOS subcounty profile workbook holds fifteen tables on a shared row index,
one row per administrative unit, four levels deep, all in a single column with
no level field. The level is carried in the cell indent, which pandas discards
because it reads values and not formatting. Reading with openpyxl keeps it, and
the hierarchy falls out directly.

The full 147 unit hierarchy is written unchanged; 03 selects from it. Sub-county
boundaries for the study district are cut from COD-AB here, and the refugee
settlement is merged into its host sub-county so that the census and the
boundaries describe the same seven units.

**Writes** `census_hierarchy.csv`, `kikuube_subcounties_codab2020.geojson`,
`kikuube_subcounty_weights.csv`, `02_urban_rural_check.csv`.

In [1]:
import sys
from pathlib import Path

# config.py sits beside the notebooks, so the working directory is enough. If a
# notebook is run from elsewhere, walk up until it is found.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
sys.path.insert(0, str(ROOT))
from config import *

import ast
import geopandas as gpd
import openpyxl
import pandas as pd

In [2]:
# COD-AB adm4 is the sub-county level, not parish. Kikuube has seven units here,
# and this is the most recent sub-county geometry published from a citable
# source (valid 2020-08-24).
assert COD_AB_SUBCOUNTIES_GJ.exists(), f"not found: {COD_AB_SUBCOUNTIES_GJ}"

k = gpd.read_file(COD_AB_SUBCOUNTIES_GJ)
k = k[k.adm2_name == DISTRICT].copy()
k["Sub_County"] = k.adm4_name.str.upper().str.strip()
k = k[["Sub_County", "adm4_pcode", "geometry"]].reset_index(drop=True)

assert len(k) == N_SUBCOUNTIES, f"expected {N_SUBCOUNTIES}, got {len(k)}"
assert k.Sub_County.is_unique, "duplicate sub-county names"

k.to_file(SUBCOUNTY_GJ, driver="GeoJSON")

print(f"{len(k)} sub-counties written to {SUBCOUNTY_GJ.name}")
print(k[["Sub_County", "adm4_pcode"]].to_string(index=False))

7 sub-counties written to kikuube_subcounties_codab2020.geojson
          Sub_County adm4_pcode
             BUGAMBE UG41180101
             BUHIMBA UG41180102
BUHIMBA TOWN COUNCIL UG41180103
             KABWOYA UG41180104
KIKUUBE TOWN COUNCIL UG41180105
        KIZIRANFUMBI UG41180106
           KYANGWALI UG41180107


## 1. Read the hierarchy from cell indent

The header in column A names the four levels in order. The first three sit at
the indent they label; PARISH is set with leading spaces rather than an indent,
so only the first three are asserted here. The level counts below confirm the
rest.

In [3]:
FIRST_DATA_ROW = 7           # rows 1 to 6 are the title and column headers
LEVELS = {0: "district", 1: "county", 2: "subcounty", 3: "parish"}
DIST = DISTRICT.upper()

wb = openpyxl.load_workbook(CENSUS_XLSX)
ws = wb["Table1"]

print("column A header")
for r in range(3, FIRST_DATA_ROW):
    cell = ws.cell(r, 1)
    raw = str(cell.value)
    print(f"  row {r}   indent {int(cell.alignment.indent or 0)}"
          f"   leading spaces {len(raw) - len(raw.lstrip())}   {raw.strip()}")

for r, expected in [(3, "district"), (4, "county"), (5, "subcounty")]:
    cell = ws.cell(r, 1)
    indent = int(cell.alignment.indent or 0)
    label = str(cell.value).strip().lower()
    assert label == expected and LEVELS[indent] == expected, \
        f"row {r} is {label} at indent {indent}, expected {expected}"

print("\nindent 0, 1 and 2 are labelled district, county and subcounty")

column A header
  row 3   indent 0   leading spaces 0   DISTRICT
  row 4   indent 1   leading spaces 0   COUNTY
  row 5   indent 2   leading spaces 0   SUBCOUNTY
  row 6   indent 2   leading spaces 3   PARISH

indent 0, 1 and 2 are labelled district, county and subcounty


In [4]:
units = []
for r in range(FIRST_DATA_ROW, ws.max_row + 1):
    cell = ws.cell(r, 1)
    if cell.value is None:
        continue
    units.append(dict(
        excel_row=r,
        name=str(cell.value).strip().upper(),
        indent=int(cell.alignment.indent or 0),
    ))

h = pd.DataFrame(units)
h["level"] = h["indent"].map(LEVELS)
assert h.level.notna().all(), \
    f"unexpected indent values {set(h[h.level.isna()].indent)}"

print(f"{len(h):,} rows read")
print(h.level.value_counts().reindex(list(LEVELS.values())).to_string())

13,521 rows read
level
district       148
county         312
subcounty     2207
parish       10854


One row at indent 0 is not a district. `National` is the country total. `APAA`,
the disputed area on the South Sudan border, is a district level entry with no
units beneath it, and is retained as such.

In [5]:
national = h[h.name == "NATIONAL"]
assert len(national) == 1, "expected exactly one National row"
national_total = national.excel_row.iloc[0]

h = h[h.name != "NATIONAL"].reset_index(drop=True)
counts = h.level.value_counts()

for level in LEVELS.values():
    print(f"  {level:10s} {counts[level]:>7,}")

childless = h[(h.level == "district") & (h.name == "APAA")]
print(f"\nAPAA is present at district level, {len(childless)} row, "
      "no counties beneath it")

assert counts["district"] == 147, f"expected 147 districts, got {counts['district']}"
assert counts["subcounty"] == 2207, f"expected 2207 subcounties, got {counts['subcounty']}"
assert counts["parish"] == 10854, f"expected 10854 parishes, got {counts['parish']}"

  district       147
  county         312
  subcounty    2,207
  parish      10,854

APAA is present at district level, 1 row, no counties beneath it


## 2. Attach parents

A unit belongs to the nearest preceding row one level up. Carrying the parent
names down the frame makes any level selectable without re-parsing.

In [6]:
current = {}
parents = {0: [], 1: [], 2: []}

for row in h.itertuples():
    current[row.indent] = row.name
    for deeper in [k for k in list(current) if k > row.indent]:
        current.pop(deeper)
    for level in parents:
        parents[level].append(current.get(level))

h["in_district"] = parents[0]
h["in_county"] = parents[1]
h["in_subcounty"] = parents[2]

assert h.in_district.notna().all(), "a unit has no district above it"

print(h[(h.in_district == DIST) & (h.level != "parish")]
      [["excel_row", "level", "name", "in_county"]].to_string(index=False))

 excel_row     level                 name            in_county
      6172  district              KIKUUBE                 None
      6173    county      BUHAGUZI COUNTY      BUHAGUZI COUNTY
      6174 subcounty              KABWOYA      BUHAGUZI COUNTY
      6180 subcounty KIKUUBE TOWN COUNCIL      BUHAGUZI COUNTY
      6185 subcounty         KIZIRANFUMBI      BUHAGUZI COUNTY
      6189 subcounty            KYANGWALI      BUHAGUZI COUNTY
      6194 subcounty        KYANGWALI RSC      BUHAGUZI COUNTY
      6224    county BUHAGUZI EAST COUNTY BUHAGUZI EAST COUNTY
      6225 subcounty              BUGAMBE BUHAGUZI EAST COUNTY
      6230 subcounty              BUHIMBA BUHAGUZI EAST COUNTY
      6236 subcounty BUHIMBA TOWN COUNCIL BUHAGUZI EAST COUNTY


## 3. Read the value tables

All fifteen tables share the row index, so a row number found in Table1 reads
the same unit in Table3 and Table12. That is asserted rather than assumed.

Table3 supplies the under five count, which weights the equity objective.
Table12 supplies households by main source of drinking water, which is the need
variable used for case selection.

In [7]:
def read_columns(sheet_name, columns):
    """Pull named numeric columns from a sheet, indexed by Excel row number."""
    sheet = wb[sheet_name]
    rows = [r for r in range(FIRST_DATA_ROW, sheet.max_row + 1)
            if sheet.cell(r, 1).value is not None]
    frame = pd.DataFrame(
        {label: [sheet.cell(r, col).value for r in rows]
         for label, col in columns.items()},
        index=pd.Index(rows, name="excel_row"),
    )
    frame.insert(0, "name",
                 [str(sheet.cell(r, 1).value).strip().upper() for r in rows])
    for label in columns:
        frame[label] = pd.to_numeric(frame[label], errors="coerce")
    return frame


t1 = read_columns("Table1", {"population": 4})
t3 = read_columns("Table3", {"p0_4": 2})
t12 = read_columns("Table12", {"unimproved": 2, "improved": 3, "households": 7})

for other, label in [(t3, "Table3"), (t12, "Table12")]:
    assert t1.index.equals(other.index), f"Table1 and {label} differ in row index"
    assert (t1["name"] == other["name"]).all(), \
        f"Table1 and {label} align on rows but not on names"
print(f"{len(t1):,} rows share an index and a name across the three sheets")

values = (t1.drop(columns="name")
            .join(t3.drop(columns="name"))
            .join(t12.drop(columns="name")))
h = h.merge(values, on="excel_row", how="left", validate="one_to_one")

for col in ("population", "p0_4", "unimproved", "improved", "households"):
    assert h[col].notna().all(), f"{col} has missing values after the join"

h["share_under5"] = h.p0_4 / h.population
h["unimproved_share"] = h.unimproved / h.households

13,521 rows share an index and a name across the three sheets


## 4. Merge the settlement and check against the boundaries

The census lists eight subcounty level units for Kikuube. The eighth, Kyangwali
RSC, is the refugee settlement, which UBOS enumerated as a special area. Kikuube
District Local Government lists seven subcounty level units and places the
settlement within Kyangwali sub-county, so it has no separate polygon in any
published boundary set. Its counts are added back to Kyangwali here, leaving
seven units that match the boundaries on name.

The merge conceals a wide internal contrast, so both units are reported before
it happens.

In [8]:
SETTLEMENT = "KYANGWALI RSC"
HOST = "KYANGWALI"

sc = (h[(h.in_district == DIST) & (h.level == "subcounty")]
      .drop(columns=["in_subcounty"])
      .rename(columns={"name": "subcounty"})
      .reset_index(drop=True))

assert len(sc) == N_SUBCOUNTIES + 1, \
    f"expected {N_SUBCOUNTIES + 1} census units, got {len(sc)}"
assert SETTLEMENT in set(sc.subcounty), f"{SETTLEMENT} not in the census tables"

# Reported before the merge, so the contrast it removes is on the record.
print("before merging")
for name, r in sc.set_index("subcounty").loc[[HOST, SETTLEMENT]].iterrows():
    print(f"  {name:16s} population {r.population:>8,.0f}   "
          f"unimproved {r.unimproved / r.households:.1%}")

COUNTS = ["population", "p0_4", "households", "unimproved", "improved"]
settlement = sc[sc.subcounty == SETTLEMENT].iloc[0]
sc.loc[sc.subcounty == HOST, COUNTS] += settlement[COUNTS].values
sc = sc[sc.subcounty != SETTLEMENT].reset_index(drop=True)

sc["share_under5"] = sc.p0_4 / sc.population
sc["unimproved_share"] = sc.unimproved / sc.households

gdf = gpd.read_file(SUBCOUNTY_GJ)
gdf["subcounty"] = gdf["Sub_County"].str.strip().str.upper()

only_census = set(sc.subcounty) - set(gdf.subcounty)
only_boundary = set(gdf.subcounty) - set(sc.subcounty)
assert not only_census and not only_boundary, \
    f"census only {only_census}, boundaries only {only_boundary}"

sc = sc.merge(gdf[["subcounty", "adm4_pcode"]], on="subcounty",
              validate="one_to_one")

merged = sc.set_index("subcounty").loc[HOST]
print(f"\nafter merging, {len(sc)} units matching the boundaries on name")
print(f"  {HOST:16s} population {merged.population:>8,.0f}   "
      f"unimproved {merged.unimproved_share:.1%}")

before merging
  KYANGWALI        population   96,556   unimproved 50.9%
  KYANGWALI RSC    population   72,098   unimproved 20.4%

after merging, 7 units matching the boundaries on name
  KYANGWALI        population  168,654   unimproved 38.2%


/var/folders/rl/8869txxn4wgcc3z1q49t4fy00000gn/T/ipykernel_32084/28602921.py:21: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[np.int64(168654)]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  sc.loc[sc.subcounty == HOST, COUNTS] += settlement[COUNTS].values
/var/folders/rl/8869txxn4wgcc3z1q49t4fy00000gn/T/ipykernel_32084/28602921.py:21: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[np.int64(28844)]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  sc.loc[sc.subcounty == HOST, COUNTS] += settlement[COUNTS].values
/var/folders/rl/8869txxn4wgcc3z1q49t4fy00000gn/T/ipykernel_32084/28602921.py:21: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[np.int64(39607)]' ha

## 5. Reconcile the subtotals

Five conservation checks, each independent of the others. Any failure means the
wrong rows were selected.

In [9]:
district_row = h[(h.level == "district") & (h.name == DIST)].iloc[0]

for label in ("population", "p0_4", "households", "unimproved"):
    parts, whole = sc[label].sum(), district_row[label]
    print(f"  {label:12s} {parts:>10,.0f}  = district  {whole:>10,.0f}")
    assert parts == whole, f"{label} does not reconcile"

assert (sc.unimproved + sc.improved == sc.households).all(), \
    "water source categories do not sum to total households"
print(f"  {'categories':12s} unimproved + improved = households in every subcounty")

national_pop = pd.to_numeric(ws.cell(national_total, 4).value)
d = h[h.level == "district"]
print(f"\n  national total {national_pop:,.0f}"
      f"  = sum of districts {d.population.sum():,.0f}")
assert d.population.sum() == national_pop, \
    "districts do not sum to the national total"

print(f"\n  national households      {d.households.sum():>12,.0f}")
print(f"  national under five      {d.p0_4.sum():>12,.0f}"
      f"   ({d.p0_4.sum() / d.population.sum():.2%})")
print(f"  national improved share  {d.improved.sum() / d.households.sum():>12.2%}")

h.to_csv(CENSUS_HIER, index=False)
print(f"\ncensus_hierarchy.csv   {len(h):,} rows, {h.shape[1]} columns")

  population      379,547  = district     379,547
  p0_4             64,066  = district      64,066
  households       92,335  = district      92,335
  unimproved       32,016  = district      32,016
  categories   unimproved + improved = households in every subcounty

  national total 45,905,417  = sum of districts 45,905,417

  national households        10,698,913
  national under five         6,778,329   (14.77%)
  national improved share        81.09%

census_hierarchy.csv   13,520 rows, 14 columns


## 6. The two subcounty weights

Both weights are constant within a subcounty, because that is the finest level
at which boundary vectors are published. Their spread differs, which bears on
whether the two objectives can diverge.

In [10]:
out = sc.sort_values("unimproved_share", ascending=False)
print(out[["subcounty", "adm4_pcode", "population", "households",
           "unimproved", "share_under5", "unimproved_share"]]
      .to_string(index=False, formatters={
          "share_under5": "{:.4f}".format,
          "unimproved_share": "{:.4f}".format}))

for label in ("share_under5", "unimproved_share"):
    lo, hi = sc[label].min(), sc[label].max()
    print(f"\n{label:18s} {lo:.1%} to {hi:.1%}, a factor of {hi / lo:.2f}")

print(f"\ndistrict unimproved share {district_row.unimproved / district_row.households:.1%}")
print(f"district under five share {district_row.p0_4 / district_row.population:.1%}")

out[["subcounty", "adm4_pcode", "population", "households", "unimproved",
     "improved", "p0_4", "share_under5", "unimproved_share"]].to_csv(
    SUBCOUNTY_WTS, index=False)
print(f"\n{SUBCOUNTY_WTS.name}   {len(out)} rows")

           subcounty adm4_pcode population households unimproved share_under5 unimproved_share
             BUGAMBE UG41180101      37951       9888       3934     0.170272         0.397856
           KYANGWALI UG41180107     168654      39607      15138     0.171025         0.382205
             KABWOYA UG41180104      70636      17238       6521     0.167846         0.378292
        KIZIRANFUMBI UG41180106      32707       8298       2863     0.163635         0.345023
BUHIMBA TOWN COUNCIL UG41180103      12615       3432        873     0.154895         0.254371
KIKUUBE TOWN COUNCIL UG41180105      17173       4315       1084     0.160659         0.251217
             BUHIMBA UG41180102      39811       9557       1603     0.171787          0.16773

share_under5       15.5% to 17.2%, a factor of 1.11

unimproved_share   16.8% to 39.8%, a factor of 2.37

district unimproved share 34.7%
district under five share 16.9%

kikuube_subcounty_weights.csv   7 rows


The under five share varies far less across the seven subcounties than the
unimproved source share does. A weight that is nearly constant makes the equity
objective close to a multiple of the efficiency objective, which is one of the
things the solved model has to be read against.

Merging the settlement into Kyangwali flattens the sharpest contrast in the
district, since the two units differ by about thirty points on the unimproved
share. The merged unit is an average over the settlement and its host
community, and results for Kyangwali should not be read as uniform within it.

Parish level values are available and vary more widely, but boundary vectors
stop at subcounty, so they cannot be assigned to grid cells.

In [11]:
par = h[(h.in_district == DIST) & (h.level == "parish")].copy()
par["share_under5"] = par.p0_4 / par.population

for label, frame in [("subcounty", sc), ("parish", par)]:
    v = frame.share_under5
    print(f"{label:10s} n {len(frame):3d}   "
          f"min {v.min():.3f}   max {v.max():.3f}   "
          f"p10 {v.quantile(.10):.3f}   p90 {v.quantile(.90):.3f}")

subcounty  n   7   min 0.155   max 0.172   p10 0.158   p90 0.171
parish     n  58   min 0.134   max 0.203   p10 0.149   p90 0.189


## 7. Where the district sits nationally

The hierarchy covers all 147 census units, so the study area's position on the
need variable can be stated rather than asserted. The units excluded from the
comparison in 03, and APAA, are left out here for the same reasons.

In [12]:
nat = h[(h.level == "district") & (~h.name.isin(EXCLUDED_DISTRICTS))].copy()
nat = nat.sort_values("unimproved_share", ascending=False).reset_index(drop=True)
nat["rank"] = nat.index + 1

row = nat[nat.name == DIST].iloc[0]
print(f"{DIST}   {row.unimproved_share:.1%} of households on an unimproved source")
print(f"  rank {int(row['rank'])} of {len(nat)}, highest need first")
print(f"  percentile {row['rank'] / len(nat):.1%}")
print(f"  national {nat.unimproved.sum() / nat.households.sum():.1%}")
print(f"  median district {nat.unimproved_share.median():.1%}")

print("\nten districts with the highest unimproved share")
print(nat.head(10)[["rank", "name", "households", "unimproved_share"]]
      .to_string(index=False, formatters={
          "unimproved_share": "{:.1%}".format, "households": "{:,.0f}".format}))

KIKUUBE   34.7% of households on an unimproved source
  rank 25 of 121, highest need first
  percentile 20.7%
  national 19.6%
  median district 21.4%

ten districts with the highest unimproved share
 rank       name households unimproved_share
    1      RAKAI     73,169            73.6%
    2       KAZO     42,433            61.7%
    3  LYANTONDE     28,627            58.0%
    4   KYEGEGWA    115,665            57.3%
    5  KALANGALA     26,564            55.7%
    6     BUVUMA     35,410            54.6%
    7 SSEMBABULE     70,473            53.3%
    8   KIRUHURA     43,180            51.3%
    9    BUHWEJU     35,305            48.1%
   10   KYENJOJO    130,853            47.1%


## 8. Is the selection indicator distorted by urban households

The indicator above covers the whole district, while the model that follows is
restricted to rural cells. Because the subcounty tabulations name town councils,
an approximate rural only version can be built by excluding them, and the two
rankings compared. The proxy is imperfect, since a town council boundary is not
a built up area, but it is the only rural split constructible from this source.

In [13]:
from scipy.stats import spearmanr

S = h[h.level == "subcounty"].copy()
S["dist"] = S.in_district
S["is_urban_unit"] = S.name.str.contains(
    "TOWN COUNCIL|MUNICIPALITY|DIVISION|CITY", case=False, na=False)

G = (S.groupby(["dist", "is_urban_unit"])
       [["population", "households", "unimproved"]].sum().unstack(fill_value=0))

D = pd.DataFrame({
    "pop_total":   G[("population", False)] + G[("population", True)],
    "pop_urban":   G[("population", True)],
    "hh_total":    G[("households", False)] + G[("households", True)],
    "unimp_total": G[("unimproved", False)] + G[("unimproved", True)],
    "hh_rural":    G[("households", False)],
    "unimp_rural": G[("unimproved", False)],
})
D = D[D.hh_rural > 0]
D["urban_share"] = D.pop_urban / D.pop_total
D["share_district"] = D.unimp_total / D.hh_total
D["share_rural"] = D.unimp_rural / D.hh_rural
D["rank_district"] = D.share_district.rank(ascending=False)
D["rank_rural"] = D.share_rural.rank(ascending=False)
D["rank_shift"] = (D.rank_district - D.rank_rural).abs()

rho = spearmanr(D.share_district, D.share_rural).statistic
print(f"districts compared            {len(D)}")
print(f"rank correlation              {rho:.4f}")
print(f"mean absolute rank shift      {D.rank_shift.mean():.1f}")
print(f"largest rank shift            {D.rank_shift.max():.0f}")
print(f"national median urban share   {D.urban_share.median():.1%}")

k = D.loc[DIST]
print(f"\n{DIST}")
print(f"  urban share of population   {k.urban_share:.1%}")
print(f"  district wide indicator     {k.share_district:.3f}  rank {k.rank_district:.0f}")
print(f"  rural only indicator        {k.share_rural:.3f}  rank {k.rank_rural:.0f}")

D.to_csv(OUT / "02_urban_rural_check.csv")

districts compared            135
rank correlation              0.9937
mean absolute rank shift      3.1
largest rank shift            21
national median urban share   23.9%

KIKUUBE
  urban share of population   7.8%
  district wide indicator     0.347  rank 27
  rural only indicator        0.355  rank 32


If the two rankings agree, the district wide indicator can be used for case
selection without claiming it measures rural access. That claim is what the
numbers above are for.